# Notebook 2 — Isolation Forest Anomaly Detection

Trains and evaluates the Isolation Forest model (Paper precision=0.88, F1=0.85).

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from src.data_generator import generate_dataset
from src.preprocessor   import PipelinePreprocessor
from src.models         import IsolationForestDetector

In [ ]:
df   = generate_dataset(seed=42)
prep = PipelinePreprocessor(window_size=10)
X    = prep.fit_transform(df)
y    = df['is_anomaly'].values
print(f'X shape: {X.shape}, anomalies: {y.sum()}')

In [ ]:
ifd = IsolationForestDetector(contamination=0.054, n_estimators=200, random_state=42)
ifd.fit(X[y == 0])
result = ifd.evaluate(X, y)
print(f"Precision: {result['precision']:.3f}  Recall: {result['recall']:.3f}  F1: {result['f1']:.3f}")

In [ ]:
scores = ifd.score_samples(X)
preds  = ifd.predict(X)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Anomaly scores
axes[0].plot(scores, color='steelblue', linewidth=0.8, label='IF Anomaly Score')
axes[0].axhline(0.5, color='red', linestyle='--', label='Decision threshold')
axes[0].scatter(np.where(y==1)[0], scores[y==1], c='red', s=50, zorder=5, label='True Anomaly')
axes[0].set(ylabel='Anomaly Score', title='Isolation Forest — Anomaly Scores')
axes[0].legend()

# Build duration coloured by prediction
axes[1].scatter(range(len(df)), df['build_duration'], c=preds, cmap='RdYlGn_r', s=10, alpha=0.8)
axes[1].set(xlabel='Pipeline Instance', ylabel='Build Duration (s)', title='Build Duration — Red=IF Flagged')

plt.tight_layout()
plt.savefig('../results/nb_isolation_forest.png', dpi=150)
plt.show()